### continuation de la carto en suivant l'exercice 3-2-2-strategie-manquants

reprise des requetes permettant la génération du dataframe

In [1]:
import duckdb
import pandas as pd
import pyarrow as py
import matplotlib.pyplot as plt

con = duckdb.connect()

In [5]:
con.sql("DESCRIBE SELECT * FROM '../../data/food.parquet'").show(max_rows=200)

┌────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│              column_name               │                                                                                                                       column_type                                                                                                                       │  null   │   key   │ default │  extra  │
│                varchar                 │                                                                                                                         varchar                                                                                                                         │ varchar │ varchar │ varchar │ varchar │
├

In [8]:
con.sql("""
    CREATE TABLE carte_rayon AS
    SELECT
        t.code,
        t.product_name[1].text AS product_name,
        NULLIF(t.brands, '') AS brands,
        t.quantity,
        t.categories_tags,
        t.compared_to_category AS main_category,
        t.food_groups_tags[1] AS rayon,
        t.countries_tags,
        t.labels_tags,
        t.stores,
        array_to_string(t.allergens_tags, ',') AS allergens,
        t.ingredients_text[1].text AS ingredients_text,
        t.nova_group,
        t.nutriscore_score,
        t.nutriscore_grade,
        t.environmental_score_grade,
        t.serving_size,
        t.completeness,
        list_filter(t.nutriments, x -> x.name = 'energy-kcal')[1]."100g" AS energy_kcal_100g,
        COALESCE(
            list_filter(t.nutriments, x -> x.name = 'energy-kj')[1]."100g",
            list_filter(t.nutriments, x -> x.name = 'energy')[1]."100g"
        ) AS energy_kj_100g,
        list_filter(t.nutriments, x -> x.name = 'fat')[1]."100g" AS fat_100g,
        list_filter(t.nutriments, x -> x.name = 'saturated-fat')[1]."100g" AS saturated_fat_100g,
        list_filter(t.nutriments, x -> x.name = 'carbohydrates')[1]."100g" AS carbohydrates_100g,
        list_filter(t.nutriments, x -> x.name = 'sugars')[1]."100g" AS sugars_100g,
        list_filter(t.nutriments, x -> x.name = 'fiber')[1]."100g" AS fiber_100g,
        list_filter(t.nutriments, x -> x.name = 'proteins')[1]."100g" AS proteins_100g,
        list_filter(t.nutriments, x -> x.name = 'salt')[1]."100g" AS salt_100g,
        list_filter(t.nutriments, x -> x.name = 'sodium')[1]."100g" AS sodium_100g,
        list_filter(t.nutriments, x -> x.name = 'fruits-vegetables-legumes')[1]."100g" AS fruits_vegetables_legumes_100g
    FROM '../../data/food.parquet' AS t
""")

In [6]:
con.sql("DROP TABLE carte_rayon")

In [ ]:
df = con.sql("SELECT * FROM carte_rayon").df()

### Stratégie de valeurs manquantes par colonne et par usage

| colonne | usage | manquants | décision | pourquoi |
|---|---|---|---|---|
| brands | affichage, substitution (même marque), assistant | 34 % | garder NA + drapeau ; « unknown » à l'affichage seulement | aucune valeur plausible à inventer |
| categories_tags | substitution, features ML (rayon), assistant | 56 % | règle métier : inclassable si vide et rayon unknown → hors périmètre | sans catégorie, pas de substitution |
| ingredients_text | assistant, features texte | 72 % | garder NA + drapeau | aucune valeur plausible à inventer |
| nova_group | affichage | 75 % | garder NA + drapeau | dérivée des ingrédients, si ingredient absent = 98.5% de chance que nova absent aussi |
| nutriscore_score/_grade | cible du modèle, affichage | 34 % | garder ; « unknown » → NA ; jamais imputer | une cible imputée est une cible inventée |
| energy_kcal_100g | score, affichage, features ML, substitution, assistant | 28 % | imputation métier : kJ / 4,184, sinon 4/4/9 ; reste → médiane par rayon calculée sur le train uniquement, après le split | dérivable avant d'être devinée |
| sugars_100g | score, features ML, affichage, assistant | 32 % | garder NA + drapeau sugars_missing, jamais imputer | une valeur inventée fausserait le score, manque MAR lié à la complétude de la fiche et au rayon |
| salt_100g / sodium_100g | score, features ML, substitution, affichage, assistant | 37 % | imputation métier : sel = sodium × 2,5 (et inverse) | relation exacte, zéro risque |
| fiber_100g | score, features ML, substitution, affichage, assistant | 62 % | garder NA + drapeau fiber_missing | MNAR : absence = souvent « pas de fibres » |
| fruits_vegetables_legumes_100g | score, features ML, subsitution, affichage, assistant | 83 % | NA + drapeau, jamais imputer | a limité aux rayons elle est indiqué, sinon traiter NA comme 0 et l'afficher |
| serving_size | sel par portion (affichage) | 68 % | a garder, affichage des valeurs pour 100g si NA | utile quand présente, jamais imputée |
| fat_100g | features ML, contrôle de cohérence, substitution, affichage, assistant | 28 % | garder NA + drapeau fat_missing ; reste → médiane par rayon calculée sur le train uniquement, après le split | MAR : suit la complétude du tableau nutritionnel ; sert de borne haute pour les saturées, reconstitution testée, regle métier atwater testé : reconstitution du fat si 3 applicable à 0,16 % des lignes seulement, gain négligeable pour une approximation, on drop |
| proteins_100g | score, features ML, affichage, substitution, assistant | 28 % | garder NA + drapeau proteins_missing ; reste → médiane par rayon calculée sur le train uniquement, après le split | MAR : manque groupé composante favorable du Nutri-Score |
| carbohydrates_100g | features ML, contrôle de cohérence, substitution, affichage, assistant | 28 % | garder NA + drapeau carbohydrates_missing ; valeurs > 100 → NA ; sert de borne haute pour sugars_100g ; jamais imputée | utile au profil nutritionnel ; une valeur inventée fausserait les contrôles de cohérence |
| saturated_fat_100g | score, features ML, affichage, substitution, assistant | 34 % | garder NA + drapeau saturated_fat_missing, règle métier : si fat_100g < 0,5 g → saturées = 0 ; sinon jamais imputer ; score non calculable si NA | composante directe du Nutri-Score, une valeur inventée fausserait le score |
| produit sans aucun nutriment clé | — | 27 % | supprimer la ligne | inutilisable pour le score et la substitution |

### Rédaction des méthodes d'implémentation des décisions : 

Vérification si les unknown sont bien présent sur le parquet aussi

In [3]:
con.sql("""
    SELECT nutriscore_grade, COUNT(*) AS n
    FROM carte_rayon
    GROUP BY 1
    ORDER BY n DESC;
    """)

┌──────────────────┬─────────┐
│ nutriscore_grade │    n    │
│     varchar      │  int64  │
├──────────────────┼─────────┤
│ unknown          │ 3110742 │
│ e                │  387545 │
│ d                │  346120 │
│ c                │  287328 │
│ a                │  200871 │
│ b                │  159205 │
│ not-applicable   │  101213 │
│ NULL             │   43447 │
└──────────────────┴─────────┘

In [3]:
con.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE nutriscore_score IS NULL) AS score_null,
        ROUND(100.0 * COUNT(*) FILTER (WHERE nutriscore_score IS NULL) / COUNT(*), 1) AS pct_null
    FROM carte_rayon
""")

┌─────────┬────────────┬──────────┐
│  total  │ score_null │ pct_null │
│  int64  │   int64    │  double  │
├─────────┼────────────┼──────────┤
│ 4636471 │    3255402 │     70.2 │
└─────────┴────────────┴──────────┘

In [10]:
con.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE nutriscore_grade IS NULL) AS score_null,
        ROUND(100.0 * COUNT(*) FILTER (WHERE nutriscore_score IS NULL) / COUNT(*), 1) AS pct_null
    FROM carte_rayon
""")

┌─────────┬────────────┬──────────┐
│  total  │ score_null │ pct_null │
│  int64  │   int64    │  double  │
├─────────┼────────────┼──────────┤
│ 4636471 │      43447 │     70.2 │
└─────────┴────────────┴──────────┘

fonction de normalisation pour nutriscore_grade, ici on remplace toute les valeurs unknown ou not-applicable par des pd.Na
la fonction retourne un dataframe nettoyé et modifié (ajout d'une colonne qui indique les grade_non_applicable en true/false) ainsi que le nombre de lignes modifier en unknow et not applicable via un dictionnaire

In [ ]:
def normalise_grade(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """1. Cible : « unknown » et « not-applicable » deviennent NA ; on ne remplit jamais."""
    res = df.copy()
    g = res["nutriscore_grade"].astype("string").str.lower()
    unknown = g.eq("unknown")
    non_appl = g.eq("not-applicable")
    res["grade_non_applicable"] = non_appl.astype("boolean")
    res["nutriscore_grade"] = res["nutriscore_grade"].mask(unknown | non_appl, pd.NA)
    return res, {"unknown": int(unknown.sum()), "not_applicable": int(non_appl.sum())}

la méthode du cours faisait une conversion des valeurs en kj de la colonne vers les kcal si ceux ci étais manquant, hors dans le parquet ce n'est pas le cas la colonne energy-kcal contient uniquement les valeurs en kcal, de ce fait on modifie la create table pour creer une nouvelle colonne energy_kj_100g qui sera la fusion des deux colonnes du parquet : energy-kj et energy

In [11]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE kcal IS NULL) AS kcal_manquants,
        COUNT(*) FILTER (WHERE kcal IS NULL AND kj IS NOT NULL) AS recuperables_kj
    FROM (
        SELECT
            list_filter(t.nutriments, x -> x.name = 'energy-kcal')[1]."100g" AS kcal,
            COALESCE(
                list_filter(t.nutriments, x -> x.name = 'energy-kj')[1]."100g",
                list_filter(t.nutriments, x -> x.name = 'energy')[1]."100g"
            ) AS kj
        FROM '../../data/food.parquet' AS t
    )
""")

┌────────────────┬─────────────────┐
│ kcal_manquants │ recuperables_kj │
│     int64      │      int64      │
├────────────────┼─────────────────┤
│        1291656 │            8011 │
└────────────────┴─────────────────┘

la valeur 900 * 4184 est la conversion en kj pour le maximum possible pour 100gr ( 100 gr de lipides = 900kcal )

In [ ]:
def imputer_energie_metier(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """2. Énergie : d'abord les kJ (exact), ensuite 4/4/9 (approximation), jamais une moyenne."""
    res = df.copy()
    kcal = res["energy_kcal_100g"]

    depuis_kj = kcal.isna() & res["energy_kj_100g"].between(0, (900 * 4.184))
    kcal = kcal.mask(depuis_kj, (res["energy_kj_100g"] / 4.184).round(1))

    calc = 4 * res["carbohydrates_100g"] + 4 * res["proteins_100g"] + 9 * res["fat_100g"]
    depuis_macros = kcal.isna() & calc.notna() & (calc <= 900)
    kcal = kcal.mask(depuis_macros, calc.round(1))

    res["energy_kcal_100g"] = kcal
    return res, {
        "depuis_kj": int(depuis_kj.sum()),
        "depuis_macros": int(depuis_macros.sum()),
        "restant": int(kcal.isna().sum()),
    }

on passe a la méthode imputer_sel_sodium, mais avant on va vérifier si il n'y as pas trop de valeurs aberrante :

In [ ]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE salt_100g < 0 OR salt_100g > 100) AS salt_outlier,
        COUNT(*) FILTER (WHERE sodium_100g < 0 OR sodium_100g > 40) AS sodium_outlier
    FROM carte_rayon
""")

┌──────────────┬─────────────────┐
│ sel_aberrant │ sodium_aberrant │
│    int64     │      int64      │
├──────────────┼─────────────────┤
│         2543 │            2637 │
└──────────────┴─────────────────┘

ok le résultat est faible et présent surement les même produits ou les deux colonnes sont renseigné mais non nul, je met un garde fou avant les application de masque

In [ ]:
def impute_salt_sodium(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """3. Salt and sodium: exact relation in both directions, plausible values only."""
    res = df.copy()
    salt, sodium = res["salt_100g"], res["sodium_100g"]
    salt_orig, sodium_orig = salt.copy(), sodium.copy()

    # garde fou pour neutraliser les valeurs aberrantes
    salt = salt.mask(~salt.between(0, 100) & salt.notna())
    sodium = sodium.mask(~sodium.between(0, 40) & sodium.notna())

    salt_ok = salt.isna() & sodium.between(0, 40)
    sodium_ok = sodium.isna() & salt.between(0, 100)
    res["salt_100g"] = salt.mask(salt_ok, sodium * 2.5)
    res["sodium_100g"] = sodium.mask(sodium_ok, salt / 2.5)

    return res, {
        "salt_from_sodium": int(salt_ok.sum()),
        "sodium_from_salt": int(sodium_ok.sum()),
        "remaining": int(res["salt_100g"].isna().sum()),
        "salt_outliers": int((~salt_orig.between(0, 100) & salt_orig.notna()).sum()),
        "sodium_outliers": int((~sodium_orig.between(0, 40) & sodium_orig.notna()).sum()),
    }

on passe a la méthode drapeau 

In [ ]:
def flag_missing(df: pd.DataFrame, column: str) -> tuple[pd.DataFrame, int]:
    """4. et 5. Fibres, marques : la valeur reste manquante, le modèle saura qu'elle manque."""
    res = df.copy()
    res[f"{column}_missing"] = res[column].isna().astype("boolean")
    return res, int(res[f"{column}_missing"].sum())

problème si la colonne brand contient des espaces ou des chaines vide, isna ne les détectera pas en tant que NULL, on vérifie

In [4]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE brands IS NULL) AS nulls,
        COUNT(*) FILTER (WHERE brands = '') AS empty_strings,
        COUNT(*) FILTER (WHERE brands <> '' AND trim(brands) = '') AS spaces_only
    FROM carte_rayon
""").show()

┌─────────┬───────────────┬─────────────┐
│  nulls  │ empty_strings │ spaces_only │
│  int64  │     int64     │    int64    │
├─────────┼───────────────┼─────────────┤
│ 1595058 │        108709 │           0 │
└─────────┴───────────────┴─────────────┘



il ya effectivement des chaines vides, je modifie la colonne brands sur le duckdb pour les transformer en null

In [9]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE brands IS NULL) AS nulls,
        COUNT(*) FILTER (WHERE brands = '') AS empty_strings,
        COUNT(*) FILTER (WHERE brands <> '' AND trim(brands) = '') AS spaces_only
    FROM carte_rayon
""").show()

┌─────────┬───────────────┬─────────────┐
│  nulls  │ empty_strings │ spaces_only │
│  int64  │     int64     │    int64    │
├─────────┼───────────────┼─────────────┤
│ 1703767 │             0 │           0 │
└─────────┴───────────────┴─────────────┘



c'est bon, on passe a la méthode mediane_par_rayon

In [ ]:
def flag_missing(df: pd.DataFrame, column: str) -> tuple[pd.DataFrame, int]:
    """4. et 5. Fibres, marques : la valeur reste manquante, le modèle saura qu'elle manque."""
    res = df.copy()
    res[f"{column}_missing"] = res[column].isna().astype("boolean")
    return res, int(res[f"{column}_missing"].sum())

ok je vais désormais rédiger les méthodes pour les colonnes qui ne sont pas dans l'exercice et celles dont avais établis de nouvelles règles, on avais établis la règle suivante si fat_100g < 0.5 alors saturated_fat_100g = 0, je lance une requete pour voir combien de lignes nutriment ne respecte pas ces règles

In [11]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE fat_100g < 0.5) AS fat_below_threshold,
        COUNT(*) FILTER (WHERE fat_100g < 0.5 AND saturated_fat_100g IS NULL) AS sat_null,
        COUNT(*) FILTER (WHERE fat_100g < 0.5 AND saturated_fat_100g = 0) AS sat_zero,
        COUNT(*) FILTER (WHERE fat_100g < 0.5 AND saturated_fat_100g > 0) AS sat_positive
    FROM carte_rayon
""").show()

┌─────────────────────┬──────────┬──────────┬──────────────┐
│ fat_below_threshold │ sat_null │ sat_zero │ sat_positive │
│        int64        │  int64   │  int64   │    int64     │
├─────────────────────┼──────────┼──────────┼──────────────┤
│              637327 │   105620 │   440532 │        91172 │
└─────────────────────┴──────────┴──────────┴──────────────┘



on a donc 91172 cas qui ne respecte pas ces règles, il faut qu'on les inspecte

In [12]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE saturated_fat_100g > fat_100g) AS sat_above_fat,
        COUNT(*) FILTER (WHERE saturated_fat_100g < 0) AS sat_negative
    FROM carte_rayon
    WHERE fat_100g < 0.5
""").show()

┌───────────────┬──────────────┐
│ sat_above_fat │ sat_negative │
│     int64     │    int64     │
├───────────────┼──────────────┤
│          2117 │            3 │
└───────────────┴──────────────┘



ici 3 valeurs négatives de sat, et 2117 valeurs qui sont supérieur a fat (incohérence), on dois les neutraliser dans la méthode

In [ ]:
def apply_saturated_fat_rule(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """Graisses saturées : bornées par le fat, et nulles si fat < 0,5 g."""
    res = df.copy()
    fat, sat = res["fat_100g"], res["saturated_fat_100g"]

    # Règle métier : NA d'origine et fat < 0,5 -> saturées = 0
    to_zero = sat.isna() & (fat < 0.5)

    # Incohérences : négatives ou supérieures au fat -> NA
    invalid = sat.notna() & ((sat < 0) | (sat > fat))
    sat = sat.mask(invalid)

    # Drapeau : NA d'origine + incohérences neutralisées, avant le passage à 0
    res["saturated_fat_100g_missing"] = sat.isna().astype("boolean")

    res["saturated_fat_100g"] = sat.mask(to_zero, 0)
    return res, {
        "set_to_zero": int(to_zero.sum()),
        "invalid_to_na": int(invalid.sum()),
    }

on avais établis une regle de reconsitution du fat dans le tableau : métier fat = (énergie − 4 × protéines − 4 × glucides) / 9 possible si les 3 sont présentes, on va vérifier si il y a des lignes ou cette règle serais utile

In [18]:
# 1. Où la reconstitution est possible et plausible
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE fat_100g IS NULL) AS fat_null,
        COUNT(*) FILTER (
            WHERE fat_100g IS NULL
              AND energy_kcal_100g BETWEEN 0 AND 900
              AND proteins_100g BETWEEN 0 AND 100
              AND carbohydrates_100g BETWEEN 0 AND 100
        ) AS reconstructible,
        COUNT(*) FILTER (
            WHERE fat_100g IS NULL
              AND energy_kcal_100g BETWEEN 0 AND 900
              AND proteins_100g BETWEEN 0 AND 100
              AND carbohydrates_100g BETWEEN 0 AND 100
              AND (energy_kcal_100g - 4 * proteins_100g - 4 * carbohydrates_100g) / 9 BETWEEN 0 AND 100
        ) AS plausible
    FROM carte_rayon
""").show()

┌──────────┬─────────────────┬───────────┐
│ fat_null │ reconstructible │ plausible │
│  int64   │      int64      │   int64   │
├──────────┼─────────────────┼───────────┤
│  1321219 │           10581 │      7619 │
└──────────┴─────────────────┴───────────┘



In [19]:
# 2. Fiabilité de la formule là où le fat est connu
con.sql("""
    SELECT
        COUNT(*) AS n,
        median(abs(fat_100g - (energy_kcal_100g - 4 * proteins_100g - 4 * carbohydrates_100g) / 9)) AS median_abs_error,
        COUNT(*) FILTER (
            WHERE abs(fat_100g - (energy_kcal_100g - 4 * proteins_100g - 4 * carbohydrates_100g) / 9) <= 1
        ) * 100.0 / COUNT(*) AS pct_within_1g
    FROM carte_rayon
    WHERE fat_100g BETWEEN 0 AND 100
      AND energy_kcal_100g BETWEEN 0 AND 900
      AND proteins_100g BETWEEN 0 AND 100
      AND carbohydrates_100g BETWEEN 0 AND 100
""").show()

┌─────────┬──────────────────┬───────────────────┐
│    n    │ median_abs_error │   pct_within_1g   │
│  int64  │      float       │      double       │
├─────────┼──────────────────┼───────────────────┤
│ 3254657 │        0.4000001 │ 73.51693281350386 │
└─────────┴──────────────────┴───────────────────┘



on drop la règle métier établi ici, le gain est faible en plus pour des approximations

on passe a la méthode qui va applique NA sur les glucides abberant ( > 100g ou negatif ), on regarde d'abord combien de lignes sont concerné

In [20]:
con.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE carbohydrates_100g IS NULL) AS carbs_null,
        COUNT(*) FILTER (WHERE carbohydrates_100g > 100) AS carbs_above_100,
        COUNT(*) FILTER (WHERE carbohydrates_100g < 0) AS carbs_negative,
        ROUND(100.0 * COUNT(*) FILTER (WHERE carbohydrates_100g > 100) / COUNT(*), 3) AS pct_above_100
    FROM carte_rayon
""").show()

┌─────────┬────────────┬─────────────────┬────────────────┬───────────────┐
│  total  │ carbs_null │ carbs_above_100 │ carbs_negative │ pct_above_100 │
│  int64  │   int64    │      int64      │     int64      │    double     │
├─────────┼────────────┼─────────────────┼────────────────┼───────────────┤
│ 4636471 │    1318649 │           14334 │              1 │         0.309 │
└─────────┴────────────┴─────────────────┴────────────────┴───────────────┘



In [ ]:
def neutralize_carbohydrates(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """Glucides : valeurs hors de [0, 100] neutralisées en NA, jamais imputées."""
    res = df.copy()
    carbs = res["carbohydrates_100g"]

    # Garde-fou : valeurs impossibles pour 100 g de produit -> NA
    invalid = carbs.notna() & ~carbs.between(0, 100)
    carbs = carbs.mask(invalid)

    # Drapeau : NA d'origine + valeurs neutralisées
    res["carbohydrates_100g_missing"] = carbs.isna().astype("boolean")

    res["carbohydrates_100g"] = carbs
    return res, {
        "invalid_to_na": int(invalid.sum()),
        "missing_total": int(carbs.isna().sum()),
    }

on avais aussi établi une régle métier qui disais que les glucides doivent être forcément supérieur au sucre ( vu que les sucres font partie des glucides ) on vérifie, mais je vérifie aussi que ces lignes ne sont pas sur des glucides aberrant

In [23]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE carbohydrates_100g NOT BETWEEN 0 AND 100) AS carbs_invalid,
        COUNT(*) FILTER (
            WHERE carbohydrates_100g NOT BETWEEN 0 AND 100
              AND sugars_100g IS NOT NULL
        ) AS carbs_invalid_with_sugars,
        COUNT(*) FILTER (
            WHERE carbohydrates_100g NOT BETWEEN 0 AND 100
              AND sugars_100g > carbohydrates_100g
        ) AS sugars_above_invalid_carbs,
        COUNT(*) FILTER (
            WHERE carbohydrates_100g NOT BETWEEN 0 AND 100
              AND sugars_100g BETWEEN 0 AND 100
        ) AS sugars_valid_with_invalid_carbs
    FROM carte_rayon
""").show()

┌───────────────┬───────────────────────────┬────────────────────────────┬─────────────────────────────────┐
│ carbs_invalid │ carbs_invalid_with_sugars │ sugars_above_invalid_carbs │ sugars_valid_with_invalid_carbs │
│     int64     │           int64           │           int64            │              int64              │
├───────────────┼───────────────────────────┼────────────────────────────┼─────────────────────────────────┤
│         14335 │                     13887 │                         26 │                            7475 │
└───────────────┴───────────────────────────┴────────────────────────────┴─────────────────────────────────┘



Je créer la méthode pour neutraliser les sucres qui ne suivent pas cette règle, on ne neutralise pas les glucides même si on ne sais pas si laquelle des deux valeurs est mauvaise 

In [ ]:
def neutralize_sugars(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """Sucres : hors de [0, 100] ou supérieurs aux glucides -> NA, jamais imputés."""
    res = df.copy()
    sugars, carbs = res["sugars_100g"], res["carbohydrates_100g"]

    # Valeurs impossibles pour 100 g
    out_of_range = sugars.notna() & ~sugars.between(0, 100)

    # Incohérence de paire : sucres > glucides (glucides déjà neutralisés si aberrants)
    above_carbs = sugars.notna() & carbs.notna() & (sugars > carbs)

    invalid = out_of_range | above_carbs
    sugars = sugars.mask(invalid)

    # Drapeau : NA d'origine + valeurs neutralisées
    res["sugars_100g_missing"] = sugars.isna().astype("boolean")

    res["sugars_100g"] = sugars
    return res, {
        "out_of_range_to_na": int(out_of_range.sum()),
        "above_carbs_to_na": int((above_carbs & ~out_of_range).sum()),
        "missing_total": int(sugars.isna().sum()),
    }